# Full grid on a rented vast.ai GPU**Turkish -> Azerbaijani transfer mechanisms** - DLE-AI-202 Track 1.Runs the frozen 114-run grid on an hourly rental. Unlike the Colab notebooks,here **every minute is billed**, so this notebook is ordered to spend thecheapest possible amount before you know whether the experiment works.## Pick the instance BEFORE you open this notebook| GPU | Runs the pinned stack? | Total (P1 x4, P2 x8) | Cost @ from-rate | Verdict ||---|---|---:|---:|---|| **H100 SXM** | yes (`sm_90`) | **~10.4 h** | **~$14** | **rent this** || H100 NVL | yes | ~10.5 h | ~$16 | fine || H100 PCIe | yes | ~10.6 h | ~$27 | 2x cost, same speed || H200 / H200 NVL | yes | ~10.4 h | ~$38-40 | extra VRAM unused || **B200 / B300** | **NO - `sm_100`** | - | $53-100 | **will not run** |`requirements.lock` pins `torch==2.4.1+cu121`. **CUDA 12.1 has no Blackwellkernels**, so B200/B300 fail with *"no kernel image is available for executionon the device."* Upgrading torch to reach them abandons the version lock thatexists so results from different machines stay comparable.Total wall-clock across every workable card spans **10.0-10.6 h** - a 4%spread - while price spans **7x**. That is because this workload is**kernel-launch-bound**, not compute-bound: sequences average 29.6 tokens, so amicro-batch is ~640 tokens and the GPU idles between tiny kernels. Measuredanchor: the RTX 4060 has ~1/4 an A100's throughput yet ran only ~1.75x slower,which puts the GPU-compute fraction at ~25%. Even an infinitely fast card capsout at a 1.33x speedup. See `docs/COMPUTE_ESTIMATES.md` section 8.## Filter the listing on vCPU and disk, not VRAM- **>= 2 vCPU per stream.** The *host* issues every kernel launch, and launches  are the bottleneck. An 8-stream run wants **>= 16 vCPU**. VRAM is not the  binding limit: at ~5.5 GB/process an 80 GB card fits 14 streams by memory and  nowhere near that by CPU.- **>= 60 GB disk.** The grid needs ~32 GB (cache 27 GB + artifacts). The  launcher refuses to start below the projection + 15% margin - it fails fast,  but only after the meter is running.- **Interruptible is genuinely safe here.** Every run writes a durable result  file and `run.skip_existing` resumes from it, so an interruption costs at most  one run. Phase 1 is resumable too - a built checkpoint is a cache hit.## Order of operationsSpend ~$4 on step 6 before committing ~$14 to step 8. If Tranche A says theconditions never escape the basin, the pre-registered rule says **stop andre-plan** - and you will have spent four dollars finding that out.

## 1 - Instance sanity check (do this first; it is free)

In [ ]:
import subprocess, os, shutil
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)
print("vCPU :", os.cpu_count())
print("disk :", f"{shutil.disk_usage('/').free/1e9:.0f} GB free")
print("""
Wanted: >= 16 vCPU for 8 streams, >= 60 GB free disk.
If the GPU says B200 or B300, DESTROY THIS INSTANCE NOW - the pinned
torch 2.4.1+cu121 has no sm_100 kernels and nothing below will run.
""")

## 2 - Code

In [ ]:
# Pick ONE. On vast.ai the usual route is a git clone or an scp'd zip.
REPO_URL  = ""     # "https://github.com/<user>/az-tokenizer-transfer.git"
LOCAL_ZIP = ""     # "/workspace/az-tokenizer-transfer.zip"  (scp it up first)

import os, shutil, subprocess, sys, zipfile
from pathlib import Path

WORK = Path("/workspace/project")
if REPO_URL:
    if WORK.exists(): shutil.rmtree(WORK)
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(WORK)], check=True)
elif LOCAL_ZIP:
    WORK.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(LOCAL_ZIP) as z: z.extractall(WORK)

if not (WORK / "configs" / "experiment.yaml").exists():
    inner = [p for p in WORK.iterdir() if p.is_dir() and (p / "configs").exists()]
    if len(inner) == 1: WORK = inner[0]
assert (WORK / "configs" / "experiment.yaml").exists(), f"no repo under {WORK}"
os.chdir(WORK); sys.path.insert(0, str(WORK))
print("project root:", WORK)

## 3 - Pinned environmentvast.ai PyTorch images ship their own torch. We install the lock file anyway:version drift between machines is exactly what invalidates a cross-machinecomparison, and this grid may be split across the course A100 and this rental.If the image's torch is already `2.4.1+cu121`, pip will no-op and this is fast.

In [ ]:
!pip install -q -r requirements.lock --extra-index-url https://download.pytorch.org/whl/cu121

In [ ]:
import torch, transformers, numpy
p = torch.cuda.get_device_properties(0)
print("torch       ", torch.__version__, "| cuda", torch.version.cuda)
print("transformers", transformers.__version__, "| numpy", numpy.__version__)
print("gpu         ", p.name, f"{p.total_memory/1e9:.0f} GB  sm_{p.major}{p.minor}")

assert torch.cuda.is_available(), "no CUDA device"
# The check that matters: can this wheel actually emit kernels for this card?
try:
    torch.zeros(8, 8, device="cuda").matmul(torch.zeros(8, 8, device="cuda"))
    torch.cuda.synchronize()
    print("\nCUDA kernel launch OK on this device.")
except RuntimeError as e:
    raise SystemExit(
        f"This GPU (sm_{p.major}{p.minor}) cannot run the pinned wheel: {e}\n"
        "Blackwell (B200/B300, sm_100) needs CUDA 12.8+/torch 2.7+, which "
        "abandons requirements.lock. Destroy the instance and rent an H100.")

## 4 - Verification gate (free, ~2 min, catches real defects)

In [ ]:
!python -c "from src.utils import verify_config_hashes; verify_config_hashes(); print('frozen config hashes OK')"
!python -m pytest tests/ -q -m "not slow" --no-header
!python -m src.training.run_grid --config configs/experiment.yaml --dry-run | head -3

## 5 - Data, transplant and controls (~1.5-2 h, mostly CPU)The OMP solve is CPU-bound, so this block does not scale with the GPU yourented - it costs roughly the same wall-clock on any card.

In [ ]:
!CONFIG=configs/experiment.yaml DEVICE=cuda bash run_all.sh --prep-only

In [ ]:
import json
s = json.load(open("results/splits.json", encoding="utf-8"))
print("splits:", s["az"]["train"], "/", s["az"]["val"], "/", s["az"]["test"],
      "| leakage:", s["leakage_check"])
assert not any(s["leakage_check"].values()), "TEST-SET LEAKAGE"

q = json.load(open("results/cross_base_transplant_quality.json", encoding="utf-8"))
for base, r in q["per_base"].items():
    t1 = r.get("top1_transplanted_canonical", {})
    print(f"{base:6s} dBPC={r.get('C1c_delta_bpc'):>9} norm_ratio={r.get('norm_ratio')} "
          f"top-1(transplanted)={t1.get('correct')}/{t1.get('masked')}")
print("\nIf both transplanted top-1 counts are ~0 on ~1,300 masked positions,")
print("the transplant has destroyed masked-LM ability. Report it - and note it")
print("weakens the xlmr zero-effect control, since a null there then cannot")
print("separate 'no deficit to fix' from 'transplant broken'.")

## 6 - Benchmark, then Tranche A (~3 h, ~$4)**This is the cheap decision point.** Do not skip it to save an hour - thewhole grid is a projection until these two cells run.

In [ ]:
!bash scripts/a100_benchmark.sh

In [ ]:
# Write the MEASURED per-process VRAM back, and derive the stream count from
# both the memory ceiling and the host's vCPU count (>= 2 vCPU per stream).
import hashlib, json, os, re
from pathlib import Path

b = json.load(open("results/hardware_benchmark.json", encoding="utf-8"))
per_proc = b["per_process_peak_gb"]

# On a machine you rented alone, the shared-workstation 10-12 GB team cap does
# not apply; the real limits are the card and the host CPU. Keep the per-process
# figure honest and report it -- see docs/COMPUTE_ESTIMATES.md section 8.7.
import torch
vram_total = torch.cuda.get_device_properties(0).total_memory / 1e9
CEILING = round(vram_total * 0.85, 1)          # leave 15% headroom
by_vram = int(CEILING // per_proc)
by_cpu  = max(1, os.cpu_count() // 2)
STREAMS = max(1, min(8, by_vram, by_cpu))

print(f"per-process peak {per_proc:.2f} GB | card {vram_total:.0f} GB "
      f"-> ceiling {CEILING} GB")
print(f"streams by VRAM {by_vram} | by vCPU {by_cpu} | capped at 8 -> USING {STREAMS}")

cfgp = Path("configs/experiment.yaml")
t = cfgp.read_text(encoding="utf-8")
t = re.sub(r"(  vram_per_stream_gb: )[\d.]+", rf"\g<1>{per_proc:.2f}", t)
t = re.sub(r"(  vram_ceiling_gb: )[\d.]+",    rf"\g<1>{CEILING}", t)
t = re.sub(r"(  parallel_streams: )\d+",      rf"\g<1>{STREAMS}", t)
t = re.sub(r"(  phase1_streams: )\d+",        rf"\g<1>{min(4, STREAMS)}", t)
cfgp.write_text(t, encoding="utf-8")

lockp = Path("configs/CONFIG_HASHES.lock")
lock = json.loads(lockp.read_text(encoding="utf-8"))
lock["configs/experiment.yaml"] = hashlib.sha256(
    cfgp.read_bytes().replace(b"\r\n", b"\n")).hexdigest()
lockp.write_text(json.dumps(lock, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
!python -c "from src.utils import verify_config_hashes; verify_config_hashes(); print('re-locked OK')"

In [ ]:
!python -m src.training.run_grid --config configs/experiment.yaml --tranche A --phase 2 --streams 1

## 7 - The decision gateFixed before the data (run plan section 8) so the outcome cannot bereinterpreted afterwards. **Read the output before running step 8.** If it saysstop, stop - and destroy the instance rather than paying for a grid that cannotanswer the question.

In [ ]:
!python -m src.analysis.aggregate --config configs/experiment.yaml
!python -m src.analysis.decompose --config configs/experiment.yaml

import json
cells = {(c["base"], c["condition"]): c
         for c in json.load(open("results/decompose.json", encoding="utf-8"))["cells"]}
baza, omp = cells[("xlm15", "baza")], cells[("xlm15", "tokenizator")]
print(f"baza        escape {baza['escape_rate_count']}  conditional F1 {baza['conditional_macro_f1']}")
print(f"tokenizator escape {omp['escape_rate_count']}  conditional F1 {omp['conditional_macro_f1']}")
b, o = baza["n_escaped"], omp["n_escaped"]
print()
if b == 0 and o == 0:
    print("NEITHER ESCAPES -> STOP AND RE-PLAN at n=10000. Do NOT tune the")
    print("optimizer. DESTROY THE INSTANCE - do not pay for the full grid.")
elif o >= 3 and b < 3:
    print("tokenizator ESCAPES, baza DOES NOT -> strong M3 evidence. Headline")
    print("result. Proceed to step 8.")
elif b >= 3 and o >= 3:
    print("BOTH ESCAPE -> pipeline works. Proceed to step 8.")
else:
    print("ERRATIC -> seed variance may dominate. Report escape rate as the")
    print("primary outcome; consider more seeds at the reference size only.")

In [ ]:
# Measured throughput -> re-project the bill before committing to it.
import json, glob, statistics
rt = [json.load(open(p, encoding="utf-8"))["az_stage_runtime_sec"]
      for p in glob.glob("results/runs/*__n=2000__*.json")]
if rt:
    per_step = statistics.median(rt) / 2000
    serial_h = 294_780 * per_step / 3600
    print(f"measured {per_step:.3f} s/optimizer-step -> serial {serial_h:.1f} h")
    for n, eff in ((1, 1.0), (2, 1.8), (4, 3.2), (8, 4.8)):
        h = serial_h / eff + 2.0
        print(f"  {n} stream(s): ~{h:5.1f} h  ->  ${h*1.33:5.0f} at $1.33/h, "
              f"${h*2.16:5.0f} at $2.16/h")

## 8 - The full gridPhase 1 builds the 30 shared Turkish checkpoints. It used to be strictlyserial (~4.3 h of unavoidable latency, ~$6-9 on a rental); it is now optionallyparallel, because Phase 1's own work list is **deduplicated by construction** -30 specs, 30 distinct cache keys, and the cache path is a SHA-256 of the spec,so no two workers can target the same path. The invariant is asserted atruntime, not assumed.Phase 2 then opens the cache **read-only**: a miss is a loud refusal, not arebuild.

In [ ]:
!python -m src.training.run_grid --config configs/experiment.yaml --phase 1 --phase1-streams {min(4, STREAMS)}

In [ ]:
import json
m = json.load(open("artifacts/tr_stage_cache/phase1_manifest.json", encoding="utf-8"))
print("sealed:", m["sealed"], "| checkpoints:", m["n_distinct_checkpoints"],
      "| workers:", m.get("phase1_streams"))
assert m["sealed"] and not [e for e in m["entries"] if e.get("status") == "FAILED"]

In [ ]:
!python -m src.training.run_grid --config configs/experiment.yaml --phase 2 --streams {STREAMS}

In [ ]:
import json
st = json.load(open("results/launcher_state_phase2.json", encoding="utf-8"))
print(f"completed {len(st['completed'])} | skipped {len(st['skipped_existing'])} "
      f"| failed {len(st['failures'])} | {st['elapsed_sec']/3600:.2f} h "
      f"| streams {st['streams_active']}")
for f in st["failures"]: print("  FAILED:", f)
print("\n" + st["timing_caveat"])

## 9 - Analysis

In [ ]:
!python -m src.analysis.aggregate --config configs/experiment.yaml
!python -m src.analysis.stats     --config configs/experiment.yaml
!python -m src.analysis.decompose --config configs/experiment.yaml
!python -m src.analysis.report    --config configs/experiment.yaml

from IPython.display import Markdown, display
display(Markdown(open("results/paper_tables.md", encoding="utf-8").read()))

In [ ]:
from IPython.display import Image, display
for n in ("escape_rate.png", "conditional_macro_f1.png"):
    print(f"figures/{n}"); display(Image(f"figures/{n}"))

## 10 - Get the results off the instance BEFORE you destroy itA destroyed vast.ai instance takes its disk with it. Download, verify thearchive opens, *then* destroy.

In [ ]:
import shutil
shutil.make_archive("/workspace/results_full", "zip", "results")
shutil.make_archive("/workspace/figures_full", "zip", "figures")
print("Download these, verify they open, THEN destroy the instance:")
!ls -lh /workspace/results_full.zip /workspace/figures_full.zip
print("\nFrom your machine:  scp -P <port> root@<host>:/workspace/results_full.zip .")

## Reporting rented compute honestlyThe brief's ~10-12 GB cap governs the **shared course workstation**, whereover-allocating starves other teams. On an instance you rented alone thatspecific harm does not apply - but the paper still has to be straight:- Report the **per-process peak** as the figure showing the work fits the  envelope. That is the honest quantity, and `streams_active` in every result  file lets a reader reconstruct it.- State the stream count and that concurrent streams ran on rented hardware.  Do **not** describe an 8-stream, ~44 GB total footprint as fitting a  10-12 GB envelope.- Quote step time from the **single-stream benchmark**, never from a contended  run. `streams_active` is recorded precisely so a contended wall-clock is  never mistaken for a run's isolated cost.- If any reported run came from rented hardware, say so in Experimental Setup  alongside the A100 window.